In order to run the following noteboooks, if you haven't done yet, you need to deploy a model that uses `text-embedding-ada-002` as base model and set the deployment name inside .env file as `AZURE_OPENAI_EMBEDDINGS_ENDPOINT`

In [ ]:
import os
import pandas as pd
import numpy as np
from openai import AzureOpenAI
from dotenv import load_dotenv

load_dotenv("/etc/.env")

client = AzureOpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],  # this is also the default, it can be omitted
  api_version = "2025-03-01-preview"
  )

#model = os.environ['AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT']
model = "text-embedding-ada-002"

SIMILARITIES_RESULTS_THRESHOLD = 0.75
## 这个文件其实模拟的就是一个向量数据库
DATASET_NAME = "../embedding_index_3m.json"

## 增加一段处理生成一段用text-embedding-3-large模型生成的embbeding，持久化到文件embedding_index_3m_3_large.json中

In [18]:
# 预处理文件
# pandas 读取 json 文件
import os
import pandas as pd
import numpy as np

DATASET_NAME = "../embedding_index_3m.json"
df = pd.read_json(DATASET_NAME)
df = df.drop(columns=['ada_v2'])

# 写入json文件，以json数组的形式
#df_to_json = df.head(10)
df.to_json("../embedding_index_3m_3_large.json", orient='records',force_ascii=False)

In [19]:
import os
import pandas as pd
import numpy as np
from openai import AzureOpenAI
from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv("/etc/.env")

client = AzureOpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],  # this is also the default, it can be omitted
  api_version = "2025-03-01-preview"
  )

model = os.environ['AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT']
#model = "text-embedding-ada-002"

## 这个文件其实模拟的就是一个向量数据库
DATASET_NAME = "../embedding_index_3m_3_large.json"

# 1. 读取原始数据集
print(f"正在加载源数据集 {DATASET_NAME}...")
df = pd.read_json(DATASET_NAME)
print(f"成功加载 {len(df)} 条记录。")

# 2. 读取数据集的summary字段
print("正在加载数据集的summary字段...")
df['summary'] = df['summary'].fillna("")
print(f"成功加载 {len(df)} 条记录。并对空值进行了填充。")

# print(df.iloc[0]['summary'])

batch_size = 16
all_embeddings = []

print(f"开始使用模型部署 '{model}' 生成 embeddings (批大小: {batch_size})...")
# 这里的 tqdm 是一个进度条库，可以让你在控制台中看到进度
for i in tqdm(range(0, len(df), batch_size), desc="生成 embeddings"):
    # 这里的 min(i + batch_size, len(df)) 确保不会超出数据集的长度
    batch_text = df.iloc[i:min(i + batch_size, len(df))]
    # 使用模型生成 embeddings
    response = client.embeddings.create(input=batch_text['summary'].tolist(), model=model)
    # 将生成的 embeddings 添加到 all_embeddings 列表中
    batch_embeddings = [item.embedding for item in response.data]
    all_embeddings.extend(batch_embeddings)


print(f"then length of all_embeddings is {len(all_embeddings)}")
df['embedding_text_3_large'] = all_embeddings


df.to_json(DATASET_NAME, orient='records', force_ascii=False)






正在加载源数据集 ../embedding_index_3m_3_large.json...
成功加载 1409 条记录。
正在加载数据集的summary字段...
成功加载 1409 条记录。并对空值进行了填充。
开始使用模型部署 'text-embedding-3-large' 生成 embeddings (批大小: 16)...


生成 embeddings:   0%|          | 0/89 [00:00<?, ?it/s]

生成 embeddings: 100%|██████████| 89/89 [01:11<00:00,  1.24it/s]


then length of all_embeddings is 1409


## 以下是用新的embedding模型实现的数据库文件

In [ ]:
import os
import pandas as pd
import numpy as np
from openai import AzureOpenAI
from dotenv import load_dotenv

load_dotenv("/etc/.env")

client = AzureOpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],  # this is also the default, it can be omitted
  api_version = "2025-03-01-preview"
  )

model = os.environ['AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT']
#model = "text-embedding-ada-002"

SIMILARITIES_RESULTS_THRESHOLD = 0.35
## 这个文件其实模拟的就是一个向量数据库
DATASET_NAME = "../embedding_index_3m_3_large.json"

Next, we are going to load the Embedding Index into a Pandas Dataframe. The Embedding Index is stored in a JSON file called `embedding_index_3m.json`. The Embedding Index contains the Embeddings for each of the YouTube transcripts up until late Oct 2023.

格式为 youtube的视频，包含了
| Speaker |title |videoid | start | seconds | summary | ada_v2

| Speaker       | title          | videoid     |start   |seconds | summary | ada_v2 |
| ------------- | -------------- | ----------- |------- |--------|-------- |------- |
| Seth Juarez, Josh Lovejoy, Sarah Bird | You're Not Solving the Problem You Think You're Solving | tJQm4mSh1s | 00:00:00 |0|Join Seth Juarez as he discusses ethical concerns with AI in this episode of the AI Show. He is joined by Josh Lovejoy, who leads Design for Microsoft Ethics & Society, and Sarah Bird, who leads Responsible AI for Cognitive Services. They explore how to think about Ethical AI and ensure that software is designed, developed, and deployed ethically. The episode emphasizes the importance of considering more than just data when it comes to ethics in machine learning.|数组(向量)|

In [35]:
def load_dataset(source: str) -> pd.core.frame.DataFrame:
    # Load the video session index
    pd_vectors = pd.read_json(source)
    # print(pd_vectors.head(5))
    return pd_vectors.drop(columns=["text"], errors="ignore").fillna("")

# 丢弃掉 text 列，对其他列中空值进行填充“”


Next, we are going to create a function called `get_videos` that will search the Embedding Index for the query. The function will return the top 5 videos that are most similar to the query. The function works as follows:

1. First, a copy of the Embedding Index is created.
2. Next, the Embedding for the query is calculated using the OpenAI Embedding API.
3. Then a new column is created in the Embedding Index called `similarity`. The `similarity` column contains the cosine similarity between the query Embedding and the Embedding for each video segment.
4. Next, the Embedding Index is filtered by the `similarity` column. The Embedding Index is filtered to only include videos that have a cosine similarity greater than or equal to 0.75.
5. Finally, the Embedding Index is sorted by the `similarity` column and the top 5 videos are returned.

In [ ]:
def cosine_similarity(a, b):
    if len(a) > len(b):
        b = np.pad(b, (0, len(a) - len(b)), 'constant')
    elif len(b) > len(a):
        a = np.pad(a, (0, len(b) - len(a)), 'constant')
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def get_videos(
    query: str, dataset: pd.core.frame.DataFrame, rows: int
) -> pd.core.frame.DataFrame:
    # create a copy of the dataset
    video_vectors = dataset.copy()

    # get the embeddings for the query    
    query_embeddings = client.embeddings.create(input=query, model=model).data[0].embedding
    print(f"query_embeddings: {query_embeddings}")

    # create a new column with the calculated similarity for each row
    # video_vectors["similarity"] = video_vectors["ada_v2"].apply(
    #     lambda x: cosine_similarity(np.array(query_embeddings), np.array(x))
    # )
    #embedding_text_3_large
    video_vectors["similarity"] = video_vectors["embedding_text_3_large"].apply(
        lambda x: cosine_similarity(np.array(query_embeddings), np.array(x))
    )

    print(f"最大的相关性是 {video_vectors['similarity'].max()}")
    # filter the videos by similarity
    mask = video_vectors["similarity"] >= SIMILARITIES_RESULTS_THRESHOLD


    video_vectors = video_vectors[mask].copy()

    # sort the videos by similarity
    video_vectors = video_vectors.sort_values(by="similarity", ascending=False).head(
        rows
    )

    # return the top rows
    return video_vectors.head(rows)

This function is very simple, it just prints out the results of the search query.

In [37]:
def display_results(videos: pd.core.frame.DataFrame, query: str):
    def _gen_yt_url(video_id: str, seconds: int) -> str:
        """convert time in format 00:00:00 to seconds"""
        return f"https://youtu.be/{video_id}?t={seconds}"

    print(f"\nVideos similar to '{query}':")
    for _, row in videos.iterrows():
        youtube_url = _gen_yt_url(row["videoId"], row["seconds"])
        print(f" - {row['title']}")
        print(f"   Summary: {' '.join(row['summary'].split()[:15])}...")
        print(f"   YouTube: {youtube_url}")
        print(f"   Similarity: {row['similarity']}")
        print(f"   Speakers: {row['speaker']}")

1. First, the Embedding Index is loaded into a Pandas Dataframe.
2. Next, the user is prompted to enter a query.
3. Then the `get_videos` function is called to search the Embedding Index for the query.
4. Finally, the `display_results` function is called to display the results to the user.
5. The user is then prompted to enter another query. This process continues until the user enters `exit`.

![](../images/notebook-search.png?WT.mc_id=academic-105485-koreyst)

You will be prompted to enter a query. Enter a query and press enter. The application will return a list of videos that are relevant to the query. The application will also return a link to the place in the video where the answer to the question is located.

Here are some queries to try out:

- What is Azure Machine Learning?
- How do convolutional neural networks work?
- What is a neural network?
- Can I use Jupyter Notebooks with Azure Machine Learning?
- What is ONNX?

In [ ]:
pd_vectors = load_dataset(DATASET_NAME)

# get user query from imput
while True:
    query = input("Enter a query: ")
    if query == "exit":
        break
    videos = get_videos(query, pd_vectors, 5)
    display_results(videos, query)

query_embeddings: [-0.02309025079011917, 0.02568790316581726, -0.00213622790761292, 0.016299894079566002, -0.0406813770532608, -0.015114999376237392, 0.00019178580259904265, 0.011393215507268906, 0.05888013914227486, 0.016482185572385788, -0.01517576351761818, 0.018168382346630096, 0.028923576697707176, -0.00598143832758069, 0.028391893953084946, -0.02164711058139801, -0.016223939135670662, -0.016816386952996254, -0.020249541848897934, -0.03293398767709732, 0.024351099506020546, -0.026432260870933533, 0.012714829295873642, 0.02555118501186371, -0.004488926846534014, -0.003326819045469165, 0.0042230854742228985, -0.010519735515117645, -0.02179902046918869, -0.009403200820088387, -0.0029755281284451485, -0.014476979151368141, 0.04180550575256348, -0.015677064657211304, -0.03779509291052818, 0.02088756300508976, -0.005670023616403341, 0.0031179434154182673, -0.009251290932297707, -0.014074419625103474, 0.02699432522058487, -0.024214381352066994, -0.037005163729190826, 0.03354162722826004,